In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
crime_path = Path("../data/raw/nypd_crime.csv")

crime_df = pd.read_csv(crime_path)

print("Rows:", len(crime_df))
print("Columns:", crime_df.columns.tolist())

Rows: 500000
Columns: ['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'loc_of_occur_desc', 'prem_typ_desc', 'juris_desc', 'jurisdiction_code', 'parks_nm', 'hadevelopt', 'housing_psa', 'x_coord_cd', 'y_coord_cd', 'susp_age_group', 'susp_race', 'susp_sex', 'transit_district', 'latitude', 'longitude', 'lat_lon', 'patrol_boro', 'station_name', 'vic_age_group', 'vic_race', 'vic_sex']


In [3]:
crime_df[
    [
        "cmplnt_fr_dt",
        "cmplnt_fr_tm",
        "ofns_desc",
        "latitude",
        "longitude"
    ]
].head()

,cmplnt_fr_dt,cmplnt_fr_tm,ofns_desc,latitude,longitude
0,2025-12-31T00:00:00.000,03:10:00,OTHER OFFENSES RELATED TO THEFT,40.639501,-74.028124
1,2025-12-31T00:00:00.000,16:30:00,HARRASSMENT 2,40.584916,-73.931174
2,2025-11-25T00:00:00.000,08:00:00,GRAND LARCENY,40.683548,-73.894247
3,2025-12-31T00:00:00.000,09:26:00,CRIMINAL MISCHIEF & RELATED OF,40.588275,-73.811348
4,2025-12-31T00:00:00.000,10:00:00,HARRASSMENT 2,40.599976,-74.006644


In [4]:
required_columns = [
    "cmplnt_fr_dt",
    "cmplnt_fr_tm",
    "latitude",
    "longitude"
]

crime_df = crime_df.dropna(subset=required_columns)

print("Rows after removing missing values:", len(crime_df))

Rows after removing missing values: 500000


In [5]:
crime_df["crime_datetime"] = pd.to_datetime(
    crime_df["cmplnt_fr_dt"].astype(str) + " " + crime_df["cmplnt_fr_tm"].astype(str),
    errors="coerce"
)

print(crime_df["crime_datetime"].head())

C:\Users\SIC\AppData\Local\Temp\ipykernel_22460\2854398484.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df["crime_datetime"] = pd.to_datetime(


0   2025-12-31 03:10:00
1   2025-12-31 16:30:00
2   2025-11-25 08:00:00
3   2025-12-31 09:26:00
4   2025-12-31 10:00:00
Name: crime_datetime, dtype: datetime64[us]


In [6]:
invalid_datetime = crime_df["crime_datetime"].isna().sum()

print("Invalid crime_datetime values:", invalid_datetime)

Invalid crime_datetime values: 0


In [7]:
crime_df[
    [
        "cmplnt_fr_dt",
        "cmplnt_fr_tm",
        "crime_datetime"
    ]
].head(10)

,cmplnt_fr_dt,cmplnt_fr_tm,crime_datetime
0,2025-12-31T00:00:00.000,03:10:00,2025-12-31 03:10:00
1,2025-12-31T00:00:00.000,16:30:00,2025-12-31 16:30:00
2,2025-11-25T00:00:00.000,08:00:00,2025-11-25 08:00:00
3,2025-12-31T00:00:00.000,09:26:00,2025-12-31 09:26:00
4,2025-12-31T00:00:00.000,10:00:00,2025-12-31 10:00:00
5,2025-12-31T00:00:00.000,12:26:00,2025-12-31 12:26:00
6,2025-11-12T00:00:00.000,10:00:00,2025-11-12 10:00:00
7,2025-12-31T00:00:00.000,03:53:00,2025-12-31 03:53:00
8,2025-12-31T00:00:00.000,11:00:00,2025-12-31 11:00:00
9,2025-12-31T00:00:00.000,23:21:00,2025-12-31 23:21:00


In [8]:
allowed_offenses = [
    "ROBBERY",
    "BURGLARY",
    "GRAND LARCENY",
    "ASSAULT",
    "PETIT LARCENY",
    "MISCHIEF"
]

print(allowed_offenses)

['ROBBERY', 'BURGLARY', 'GRAND LARCENY', 'ASSAULT', 'PETIT LARCENY', 'MISCHIEF']


In [9]:
crime_df = crime_df[
    crime_df["ofns_desc"].isin(allowed_offenses)
].copy()

print("Rows after offense filtering:", len(crime_df))

Rows after offense filtering: 156899


In [10]:
crime_df["ofns_desc"].value_counts()

ofns_desc
PETIT LARCENY    90616
GRAND LARCENY    42166
ROBBERY          13222
BURGLARY         10895
Name: count, dtype: int64

In [11]:
print("Unique offense categories:", crime_df["ofns_desc"].unique())

Unique offense categories: <ArrowStringArray>
['GRAND LARCENY', 'PETIT LARCENY', 'ROBBERY', 'BURGLARY']
Length: 4, dtype: str


In [12]:
crime_df["ofns_desc"].value_counts().head(30)

ofns_desc
PETIT LARCENY    90616
GRAND LARCENY    42166
ROBBERY          13222
BURGLARY         10895
Name: count, dtype: int64

In [13]:
print(
    crime_df[crime_df["ofns_desc"].isin(["ASSAULT", "MISCHIEF"])]["ofns_desc"]
    .value_counts()
)

Series([], Name: count, dtype: int64)


In [14]:
print("ALL RAW OFFENSE CATEGORIES:")
print(crime_df["ofns_desc"].value_counts())

ALL RAW OFFENSE CATEGORIES:
ofns_desc
PETIT LARCENY    90616
GRAND LARCENY    42166
ROBBERY          13222
BURGLARY         10895
Name: count, dtype: int64


In [15]:
crime_df[["cmplnt_fr_dt", "cmplnt_fr_tm", "crime_datetime"]].head()

,cmplnt_fr_dt,cmplnt_fr_tm,crime_datetime
2,2025-11-25T00:00:00.000,08:00:00,2025-11-25 08:00:00
6,2025-11-12T00:00:00.000,10:00:00,2025-11-12 10:00:00
7,2025-12-31T00:00:00.000,03:53:00,2025-12-31 03:53:00
8,2025-12-31T00:00:00.000,11:00:00,2025-12-31 11:00:00
13,2025-12-30T00:00:00.000,16:00:00,2025-12-30 16:00:00


In [16]:
crime_df["crime_time"] = crime_df["crime_datetime"].dt.time

crime_df["is_nighttime"] = (
    (crime_df["crime_datetime"].dt.hour >= 18) |
    (crime_df["crime_datetime"].dt.hour <= 6)
)

crime_df["is_nighttime"].value_counts()

is_nighttime
False    93703
True     63196
Name: count, dtype: int64

In [17]:
crime_df = crime_df[
    crime_df["is_nighttime"] == True
].copy()

print("Rows after nighttime filtering:", len(crime_df))

Rows after nighttime filtering: 63196


In [18]:
crime_df["is_nighttime"].value_counts()

is_nighttime
True    63196
Name: count, dtype: int64

In [19]:
crime_df = crime_df[
    crime_df["latitude"].between(40.49, 40.92)
    &
    crime_df["longitude"].between(-74.26, -73.69)
].copy()

print("Rows after NYC coordinate filtering:", len(crime_df))

Rows after NYC coordinate filtering: 63196


In [20]:
print("Latitude range:")
print(crime_df["latitude"].min(), crime_df["latitude"].max())

print("\nLongitude range:")
print(crime_df["longitude"].min(), crime_df["longitude"].max())

Latitude range:
40.499625 40.912722

Longitude range:
-74.254439 -73.700719


In [21]:
crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(
        crime_df["longitude"],
        crime_df["latitude"]
    ),
    crs="EPSG:4326"
)

In [22]:
print(crime_gdf.head())

   cmplnt_num             cmplnt_fr_dt cmplnt_fr_tm             cmplnt_to_dt  \
7   318185372  2025-12-31T00:00:00.000     03:53:00  2025-12-31T00:00:00.000   
14  318194633  2025-12-30T00:00:00.000     20:00:00  2025-12-31T00:00:00.000   
22  318205065  2025-12-31T00:00:00.000     18:00:00  2025-12-31T00:00:00.000   
24  318198685  2025-12-13T00:00:00.000     02:40:00  2025-12-13T00:00:00.000   
32  318185349  2025-12-30T00:00:00.000     22:30:00  2025-12-31T00:00:00.000   

   cmplnt_to_tm  addr_pct_cd                   rpt_dt  ky_cd      ofns_desc  \
7      04:25:00          106  2025-12-31T00:00:00.000    341  PETIT LARCENY   
14     14:00:00           67  2025-12-31T00:00:00.000    109  GRAND LARCENY   
22     18:10:00          121  2025-12-31T00:00:00.000    109  GRAND LARCENY   
24     03:11:00           81  2025-12-31T00:00:00.000    109  GRAND LARCENY   
32     08:00:00           71  2025-12-31T00:00:00.000    341  PETIT LARCENY   

    pd_cd  ...                      lat_lon 

In [23]:
print(crime_gdf.crs)

EPSG:4326


In [24]:
print(crime_gdf.geometry.head())

7     POINT (-73.82417 40.68623)
14    POINT (-73.94478 40.64234)
22    POINT (-74.16907 40.58231)
24     POINT (-73.9297 40.69386)
32      POINT (-73.9488 40.6572)
Name: geometry, dtype: geometry


In [25]:
print("Final rows:", len(crime_gdf))

print("\nOffense categories:")
print(crime_gdf["ofns_desc"].value_counts())

print("\nNighttime values:")
print(crime_gdf["is_nighttime"].value_counts())

print("\nMissing values:")
print(
    crime_gdf[
        ["crime_datetime", "latitude", "longitude", "ofns_desc"]
    ].isna().sum()
)

print("\nCRS:")
print(crime_gdf.crs)

Final rows: 63196

Offense categories:
ofns_desc
PETIT LARCENY    31476
GRAND LARCENY    18256
ROBBERY           7409
BURGLARY          6055
Name: count, dtype: int64

Nighttime values:
is_nighttime
True    63196
Name: count, dtype: int64

Missing values:
crime_datetime    0
latitude          0
longitude         0
ofns_desc         0
dtype: int64

CRS:
EPSG:4326


In [26]:
crime_gdf.to_parquet(
    "../data/processed/clean_crime.parquet",
    index=False
)

In [27]:
# Verification

test_crime = gpd.read_parquet(
    "../data/processed/clean_crime.parquet"
)

print(test_crime.shape)
print(test_crime.head())
print(test_crime.crs)


(63196, 39)
  cmplnt_num             cmplnt_fr_dt cmplnt_fr_tm             cmplnt_to_dt  \
0  318185372  2025-12-31T00:00:00.000     03:53:00  2025-12-31T00:00:00.000   
1  318194633  2025-12-30T00:00:00.000     20:00:00  2025-12-31T00:00:00.000   
2  318205065  2025-12-31T00:00:00.000     18:00:00  2025-12-31T00:00:00.000   
3  318198685  2025-12-13T00:00:00.000     02:40:00  2025-12-13T00:00:00.000   
4  318185349  2025-12-30T00:00:00.000     22:30:00  2025-12-31T00:00:00.000   

  cmplnt_to_tm  addr_pct_cd                   rpt_dt  ky_cd      ofns_desc  \
0     04:25:00          106  2025-12-31T00:00:00.000    341  PETIT LARCENY   
1     14:00:00           67  2025-12-31T00:00:00.000    109  GRAND LARCENY   
2     18:10:00          121  2025-12-31T00:00:00.000    109  GRAND LARCENY   
3     03:11:00           81  2025-12-31T00:00:00.000    109  GRAND LARCENY   
4     08:00:00           71  2025-12-31T00:00:00.000    341  PETIT LARCENY   

   pd_cd  ...                      lat_lon  

In [28]:
print(
    crime_df[
        ["cmplnt_fr_dt", "cmplnt_fr_tm"]
    ].head(20).to_string(index=False)
)

           cmplnt_fr_dt cmplnt_fr_tm
2025-12-31T00:00:00.000     03:53:00
2025-12-30T00:00:00.000     20:00:00
2025-12-31T00:00:00.000     18:00:00
2025-12-13T00:00:00.000     02:40:00
2025-12-30T00:00:00.000     22:30:00
2025-12-31T00:00:00.000     01:35:00
2025-12-29T00:00:00.000     20:00:00
2025-12-31T00:00:00.000     22:45:00
2025-12-21T00:00:00.000     18:00:00
2025-12-31T00:00:00.000     06:24:00
2025-12-30T00:00:00.000     19:30:00
2025-07-19T00:00:00.000     00:01:00
2025-12-11T00:00:00.000     06:30:00
2025-12-27T00:00:00.000     18:57:00
2025-12-31T00:00:00.000     21:30:00
2025-12-31T00:00:00.000     21:40:00
2025-12-31T00:00:00.000     21:13:00
2025-12-29T00:00:00.000     05:49:00
2025-12-12T00:00:00.000     22:54:00
2025-12-30T00:00:00.000     18:00:00


In [29]:
print(crime_df["cmplnt_fr_dt"].head(20).tolist())

['2025-12-31T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-13T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-21T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-30T00:00:00.000', '2025-07-19T00:00:00.000', '2025-12-11T00:00:00.000', '2025-12-27T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-12T00:00:00.000', '2025-12-30T00:00:00.000']


In [30]:
print(crime_df["cmplnt_fr_tm"].head(20).tolist())

['03:53:00', '20:00:00', '18:00:00', '02:40:00', '22:30:00', '01:35:00', '20:00:00', '22:45:00', '18:00:00', '06:24:00', '19:30:00', '00:01:00', '06:30:00', '18:57:00', '21:30:00', '21:40:00', '21:13:00', '05:49:00', '22:54:00', '18:00:00']


In [31]:
print("Minimum date:", crime_df["cmplnt_fr_dt"].min())
print("Maximum date:", crime_df["cmplnt_fr_dt"].max())

Minimum date: 1015-05-01T00:00:00.000
Maximum date: 2025-12-31T00:00:00.000


In [32]:
import os

print(
    "CSV size in MB:",
    round(
        os.path.getsize("../data/raw/nypd_crime.csv") / (1024 * 1024),
        2
    )
)

CSV size in MB: 213.08


In [33]:
print("Rows in crime_df:", len(crime_df))

Rows in crime_df: 63196


In [34]:
raw_check = pd.read_csv(
    "../data/raw/nypd_crime.csv",
    usecols=["cmplnt_fr_dt", "cmplnt_fr_tm"]
)

print(raw_check.dtypes)

print("\nFirst 20 rows:")
print(raw_check.head(20).to_string(index=False))

cmplnt_fr_dt    str
cmplnt_fr_tm    str
dtype: object

First 20 rows:
           cmplnt_fr_dt cmplnt_fr_tm
2025-12-31T00:00:00.000     03:10:00
2025-12-31T00:00:00.000     16:30:00
2025-11-25T00:00:00.000     08:00:00
2025-12-31T00:00:00.000     09:26:00
2025-12-31T00:00:00.000     10:00:00
2025-12-31T00:00:00.000     12:26:00
2025-11-12T00:00:00.000     10:00:00
2025-12-31T00:00:00.000     03:53:00
2025-12-31T00:00:00.000     11:00:00
2025-12-31T00:00:00.000     23:21:00
2025-12-31T00:00:00.000     11:31:00
2025-12-30T00:00:00.000     18:30:00
2025-12-31T00:00:00.000     22:20:00
2025-12-30T00:00:00.000     16:00:00
2025-12-30T00:00:00.000     20:00:00
2025-12-30T00:00:00.000     14:45:00
2025-12-31T00:00:00.000     16:45:00
2025-12-31T00:00:00.000     14:30:00
2025-12-31T00:00:00.000     11:00:00
2025-12-26T00:00:00.000     10:30:00


In [35]:
raw_check["cmplnt_fr_dt_parsed"] = pd.to_datetime(
    raw_check["cmplnt_fr_dt"],
    errors="coerce",
    format="mixed"
)

print(
    raw_check["cmplnt_fr_dt_parsed"]
    .sort_values()
    .head(20)
)

392498   1014-04-20
390947   1015-05-01
108744   1015-09-02
74922    1015-11-09
68729    1015-11-14
33608    1015-12-06
384124   1025-05-05
69904    1025-11-13
23548    1025-12-06
25882    1025-12-12
16446    1025-12-19
328374   1963-05-01
466490   1970-01-01
465873   1975-03-16
386084   1975-04-18
352352   1975-05-24
334999   1975-06-04
206037   1975-06-20
204590   1975-08-23
154383   1975-08-30
Name: cmplnt_fr_dt_parsed, dtype: datetime64[us]


In [36]:
raw_check[
    raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
].head(20)

,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_fr_dt_parsed
16446,1025-12-19T00:00:00.000,05:35:00,1025-12-19
23548,1025-12-06T00:00:00.000,20:15:00,1025-12-06
25882,1025-12-12T00:00:00.000,15:38:00,1025-12-12
33608,1015-12-06T00:00:00.000,18:00:00,1015-12-06
68729,1015-11-14T00:00:00.000,13:48:00,1015-11-14
69904,1025-11-13T00:00:00.000,08:30:00,1025-11-13
74922,1015-11-09T00:00:00.000,20:00:00,1015-11-09
108744,1015-09-02T00:00:00.000,08:00:00,1015-09-02
384124,1025-05-05T00:00:00.000,15:02:00,1025-05-05
390947,1015-05-01T00:00:00.000,20:50:00,1015-05-01


In [37]:
print(
    "Number of dates before 1900:",
    (
        raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
    ).sum()
)

Number of dates before 1900: 11


In [38]:
print(
    "Raw minimum:",
    raw_check["cmplnt_fr_dt_parsed"].min()
)

print(
    "Raw maximum:",
    raw_check["cmplnt_fr_dt_parsed"].max()
)

Raw minimum: 1014-04-20 00:00:00
Raw maximum: 2025-12-31 00:00:00


In [39]:
print("Raw CSV rows:", len(raw_check))

Raw CSV rows: 500000


In [40]:
print(raw_check.dtypes)

cmplnt_fr_dt                      str
cmplnt_fr_tm                      str
cmplnt_fr_dt_parsed    datetime64[us]
dtype: object


In [41]:
print(
    raw_check[
        raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
    ].head(20)
)

                   cmplnt_fr_dt cmplnt_fr_tm cmplnt_fr_dt_parsed
16446   1025-12-19T00:00:00.000     05:35:00          1025-12-19
23548   1025-12-06T00:00:00.000     20:15:00          1025-12-06
25882   1025-12-12T00:00:00.000     15:38:00          1025-12-12
33608   1015-12-06T00:00:00.000     18:00:00          1015-12-06
68729   1015-11-14T00:00:00.000     13:48:00          1015-11-14
69904   1025-11-13T00:00:00.000     08:30:00          1025-11-13
74922   1015-11-09T00:00:00.000     20:00:00          1015-11-09
108744  1015-09-02T00:00:00.000     08:00:00          1015-09-02
384124  1025-05-05T00:00:00.000     15:02:00          1025-05-05
390947  1015-05-01T00:00:00.000     20:50:00          1015-05-01
392498  1014-04-20T00:00:00.000     19:49:00          1014-04-20


In [42]:
print("Raw CSV rows:", len(raw_check))
print("Raw minimum:", raw_check["cmplnt_fr_dt_parsed"].min())
print("Raw maximum:", raw_check["cmplnt_fr_dt_parsed"].max())

Raw CSV rows: 500000
Raw minimum: 1014-04-20 00:00:00
Raw maximum: 2025-12-31 00:00:00
